In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from simsopt.mhd import Vmec

from spectre import SPECTRE, SPECTREout
from spectre.converters import simsvmec2spectre

%matplotlib inline

## Load VMEC equilibria

Two sources can be used:
- `input.<name>` — VMEC input file. Contains profile info and the plasma boundary only.
- `wout_<name>.nc` — VMEC output file. Contains interfaces of the solved equilibrium. Generate this by running VMEC or VMEC++ on the input file. Uncomment the lines if you have generated this file. 

In [ ]:
vmec_in  = Vmec("input.iota")


## Convert from `input.iota` (no interface guesses)

`linitialize=1` — SPECTRE linearly interpolates the initial interface geometry from the boundary.

In [ ]:
tfluxes = np.linspace(0.0, 1.0, 6, endpoint=True)[1:]  # 5 volumes

params_from_input = simsvmec2spectre(vmec_in, tfluxes=tfluxes, mpol=6, ntor=5, interface_guess=None)

The input without good interface guesses interpolates from the boundary. It has self-intersections and will not work. 

In [ ]:
spectre_input = SPECTRE.from_input_params(params_from_input)
out_from_input = spectre_input.create_spectreout()
out_from_input.plot_infaces()

## Create interface guesses using `map2disc`

You can greate a better initial start by computing the interfaces using `Map2Disc`, which ensures that the interfaces contain no self-intersections.`Map2Disc` routines can take a bit longer for high resolution interfaces. 

In [ ]:
params_map = simsvmec2spectre(vmec_in, tfluxes=tfluxes, mpol=6, ntor=5, plot=False, interface_guess='map2disc')

In [ ]:
spectre_map = SPECTRE.from_input_params(params_map)
out_from_map = spectre_map.create_spectreout()
out_from_map.plot_infaces()

## Convert from `wout_iota.nc` (with interface guesses)

If we have a `simsopt.mhd.Vmec` object generated from a wout file (run vmec on input.iota to generate this), we can directly read the interfaces computed there, giving us an even closer approximation of the initial equilibrium.  

You can also call vmec_in.run(), and the wout info will be present in the vmec_in object. 

Uncomment the following code once you have generated the Wout file to generate interfaces from it. 

In [ ]:
#vmec_wout = Vmec("wout_iota.nc")
#params_from_wout = simsvmec2spectre(vmec_wout, tfluxes=tfluxes, mpol=6, ntor=5, plot=False, interface_guess='wout')


In [ ]:
#spectre_wout = SPECTRE.from_input_params(params_from_wout)
#out_from_wout = spectre_wout.create_spectreout()
#out_from_wout.plot_infaces()

In [ ]:
# Save the input parameters to a TOML file

from spectre.file_io.toml_io import write_input_parameters_to_toml

write_input_parameters_to_toml(
    params_map, "input.toml"
)
